In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader




# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28,28)), # TODO: Resize to 28x28
    transforms.Grayscale(3),# Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),# TODO: Convert to Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])


# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'



class_to_idx = {cls_name: idx for idx, cls_name in enumerate(letters)}
print(class_to_idx)
train_dataloader = DataLoader(train_dataset, 16, shuffle = True)
test_dataloader = DataLoader(test_dataset, 16, shuffle = False)
# Create DataLoaders and display samples
# Write your code here

import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
data_iter = iter(train_dataloader)
images, labels = next(data_iter)



# Show images
import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

model = efficientnet_v2_s(pretrained = True)
# Write your code here


# Freeze ALL backbone layers
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

# Replace classifier head (this will be trainable by default)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, num_classes)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model


In [ ]:
import torch.optim as optim


def train_one_epoch(model, optimizer, criterion,train_loader, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total_samples = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        outputs = outputs-1
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+= loss.item()

        predictions = outputs.argmax(dim=1)
        correct+= (predictions == labels).sum().item()
        total_samples += labels.shape[0]

    avg_loss = total_loss/len(train_loader)
    accuracy = correct/total_samples
    return avg_loss, accuracy


def validate_one_epoch(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0
    with torch.no_grad():  # Disable gradient calculation

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)
            correct+= (predictions == labels).sum().item()
            total_samples += labels.shape[0]

            total_loss+= loss.item()

    avg_loss = total_loss/len(test_loader)
    accuracy = correct/total_samples

    return avg_loss, accuracy

In [ ]:
# Write your code here

import torch
import torch.nn as nn
from torchvision import models



# Replace the classifier (fc) to match your number of classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()



model = model.to(device)
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)


In [ ]:
# TO DO

num_epochs = 5

print(device)
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
for epoch in range(num_epochs):
    avg_loss_train, train_accuracy = train_one_epoch(model, optimizer, criterion, train_dataloader, device)

    avg_loss_validation, val_accuracy = validate_one_epoch(model, criterion, val_dataloader, device)
    train_losses.append(avg_loss_train)
    train_accuracies.append(train_accuracy)
    val_losses.append(avg_loss_validation)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss_train:.4f}, Train accuracy: {train_accuracy:.4f} ,Val Loss: {avg_loss_validation:.4f}, Val Accuracy: {val_accuracy:.4f}')


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
